# OLS — Linear Regression (Turkey, Pipeline B)

**Model**: `sklearn.linear_model.LinearRegression`  
**Target**: `ngdprsaxdctrq` (quarterly log-diff real GDP, Turkey)  
**Variables**: Cat2 — ipi_sa, usd_try_avg, cpi_sa, fin_acc + 3 COVID = 7 total  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: No / **HP tuning**: No / **n_lags**: 3

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
VAL_END     = '2017-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
N_LAGS      = 3
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0}

print('OLS (Turkey) — Cat2 variables, n_lags={}, scaling=NO'.format(N_LAGS))

OLS (Turkey) — Cat2 variables, n_lags=3, scaling=NO


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat2', with_covid=True)
print('Features: {} -> {}'.format(len(features), features))
print('Target: {}'.format(TARGET))

Features: 7 -> ['ipi_sa', 'usd_try_avg', 'cpi_sa', 'fin_acc', 'covid_2020q2', 'covid_2020q3', 'covid_2020q4']
Target: ngdprsaxdctrq


In [3]:
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions  = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)
        train_mask = (monthly['date'] >= TRAIN_START) & (monthly['date'] <= forecast_date)
        train = monthly[train_mask].reset_index(drop=True)

        train_masked = gen_lagged_data(metadata, train.copy(), forecast_date, lag=0)
        train_filled = mean_fill_dataset(train_masked, train_masked)
        train_flat   = flatten_data(train_filled, TARGET, N_LAGS)
        train_flat   = train_flat.loc[train_flat.date.dt.month.isin([3, 6, 9, 12]), :]
        train_flat   = train_flat.dropna(axis=0, how='any').reset_index(drop=True)

        feature_cols = [
            c for c in train_flat.columns
            if c != 'date' and c != TARGET
            and any(c == f or c.startswith(f + '_') for f in features)
        ]

        if len(train_flat) < 10:
            predictions[vintage_name].append(np.nan)
            continue

        X_train = train_flat[feature_cols].values
        y_train = train_flat[TARGET].values

        try:
            model = LinearRegression()
            model.fit(X_train, y_train)

            # Build test row: context (N_LAGS recent rows) + current forecast_date row
            test_row    = monthly[monthly['date'] == forecast_date].reset_index(drop=True)
            test_masked = gen_lagged_data(metadata, test_row.copy(), forecast_date, lag=0)
            test_filled = mean_fill_dataset(train_masked, test_masked)
            ctx    = train_filled.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb    = pd.concat([ctx, test_filled], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            test_flat = cmb_fl.tail(1)
            X_test = test_flat[feature_cols].values
            pred   = model.predict(X_test)[0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vintage_name].append(pred)

    if (i + 1) % 8 == 0:
        print('  {} / {} quarters'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

  32 / 32 quarters
Done. 32 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        'date': test_quarters, 'actual': actuals_list,
        'prediction': predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, 'ols_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))

Saved ols_m1.csv (32 rows) pred=[-0.0203,0.0958]
Saved ols_m2.csv (32 rows) pred=[-0.0419,0.2031]
Saved ols_m3.csv (32 rows) pred=[-0.0001,0.0235]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print('OLS (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))

OLS (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.020740  MAE=0.013984  N=8
  m2  RMSFE=0.015767  MAE=0.014316  N=8
  m3  RMSFE=0.017716  MAE=0.013730  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.059789  MAE=0.040942  N=7
  m2  RMSFE=0.040077  MAE=0.027364  N=7
  m3  RMSFE=0.069465  MAE=0.043668  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.040196  MAE=0.030323  N=16
  m2  RMSFE=0.027935  MAE=0.022838  N=16
  m3  RMSFE=0.011720  MAE=0.007951  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.041272  MAE=0.028047  N=32
  m2  RMSFE=0.028640  MAE=0.021704  N=32
  m3  RMSFE=0.034773  MAE=0.017411  N=32
